# How Do Synapses Learn?

## A question from the previous labs

In the degeneracy lab you discovered that many different sets of synaptic weights can all produce a valid pyloric rhythm. This raised a new question:

> **If synaptic weights can vary so widely — how does the brain settle on any particular set? And once it does, how does it keep them stable over time?**

The answer involves **synaptic plasticity** — the ability of synapses to change their strength based on the activity of the neurons they connect.

This is one of the most important principles in neuroscience. It underlies learning and memory, sensory adaptation, and — as you will see — the long-term stability of circuits like the STG.

In this notebook you will explore two forms of synaptic plasticity through simulation — and discover a fundamental principle that has shaped how neuroscientists think about learning in the brain.

> 💡 Write your answers directly into this notebook. You will submit it as a PDF at the end.

Run the cell below to set up the environment.

In [1]:
#@title ▶ Run to initialize lab environment
#@markdown The code is hidden by default.
!pip install ipympl ipywidgets stg-net -q

import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import display

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

from stg_net.neuron import LIF
from stg_net.input import Poisson_generator, Gaussian_generator, Current_injector
from stg_net.conn import Simulator
from stg_net.helper import plot_volt_trace

plt.rcParams.update({
    'axes.labelsize': 13, 'axes.titlesize': 13,
    'font.size': 13, 'legend.fontsize': 10,
    'lines.markersize': 7., 'lines.linewidth': 2.,
    'xtick.labelsize': 11, 'ytick.labelsize': 11
})
%matplotlib inline

# Shared neuron types
tonic_neuron    = {'tau_m':20., 'a':0.,    'tau_w':30.,  'b':3.,  'V_reset':-55.}
adapting_neuron = {'tau_m':20., 'a':0.,    'tau_w':100., 'b':0.5, 'V_reset':-55.}
bursting_neuron = {'tau_m':5.,  'a':0.,    'tau_w':100., 'b':1.,  'V_reset':-46.}
irregular_neuron= {'tau_m':10., 'a':-0.01, 'tau_w':50.,  'b':1.2, 'V_reset':-46.}

T, dt = 1e3, 0.1
N     = 2
Itypes = ['Icur', 'Gaussian', 'Poisson']
print('Environment ready.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.0/519.0 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 29.4 MB/s eta 0:00:00
Environment ready.


## Part 1: Synapses that change with activity

### What is synaptic plasticity?

A synapse is not a fixed wire. Its strength — how strongly the presynaptic neuron influences the postsynaptic neuron — can increase or decrease depending on how the two neurons are being used.

This was first articulated by Donald Hebb in 1949, long before anyone could measure it directly:

> *"When an axon of cell A is near enough to excite cell B and repeatedly or persistently takes part in firing it, some growth process or metabolic change takes place in one or both cells such that A's efficiency, as one of the cells firing B, is increased."*
>
> — Donald Hebb, *The Organization of Behavior* (1949)

This is often summarised as: **"neurons that fire together, wire together."**

The simplest version of this rule says that the synaptic weight between two neurons changes in proportion to how often both neurons are active at the same time. More activity in both → stronger synapse. Less activity → weaker synapse.

---

### Reading the simulation output

Before running anything, make sure you understand what you will see:

**Left panel — Raster plot:**
- **Blue (Pre)** — the presynaptic neuron: the one sending signals
- **Red (Post)** — the postsynaptic neuron: the one receiving signals
- The rate shown in the legend is each neuron's firing rate in Hz

**Right panel — Weight trace:**
- The **purple line** shows the synaptic weight between Pre and Post over time
- If it rises → the synapse is getting stronger (potentiation)
- If it falls → the synapse is getting weaker (depression)
- Blue and red ticks along the bottom show individual spike times — you can see the weight jump or dip with each spike
- **J_ts** sets the initial weight when the simulation starts

**The sliders:**
- **J_ts** — starting synaptic weight
- **rt** — input rate to the presynaptic neuron (Pre, blue)
- **rs** — input rate to the postsynaptic neuron (Post, red)
- **Itype** — type of input (constant, noisy, or spike-based)

---

### Predict — before touching the sliders

*According to Hebb's rule, what should happen to the synaptic weight when both Pre and Post are firing at high rates?*

**My prediction:**
```
[Write here]
```

*What should happen when Pre fires at a high rate but Post fires at a low rate?*

**My prediction:**
```
[Write here]
```

*What do you think happens to the weight if both neurons are silent?*

**My prediction:**
```
[Write here]
```

In [2]:
#@title ▶ Run to start rate-dependent plasticity simulation
#@markdown The code is hidden by default.

def update_rate_plasticity(J_ts=1.0, rt=50., rs=50., Itype='Icur'):
    h    = Simulator(dt=dt)
    nrns = [LIF(sim=h) for _ in range(N)]
    for nrn in nrns:
        nrn.update(tonic_neuron)

    if Itype == 'Icur':
        noises = [Current_injector(sim=h, rate=r) for r in [rt, rs]]
    elif Itype == 'Gaussian':
        noises = [Gaussian_generator(sim=h, mean=r, std=r) for r in [rt, rs]]
    elif Itype == 'Poisson':
        noises = [Poisson_generator(sim=h, rate=r*3) for r in [rt, rs]]

    for noise, nrn in zip(noises, nrns):
        nrn.connect(noise, {'ctype':'Static', 'weight':1e0, 'delay':5})

    tps       = [['Static']*N for _ in range(N)]
    tps[0][1] = 'Hebb'
    con       = np.array([[0., J_ts], [0., 0.]])
    dly       = np.random.uniform(2., 5., (N, N))
    synspecs  = [[{'ctype':tps[i][j], 'weight':con[i,j], 'delay':dly[i,j]}
                  for j in range(N)] for i in range(N)]
    cons = h.connect(nrns, nrns, synspecs)
    h.run(T)

    rate_pre  = len(nrns[1].spikes['times']) / T * 1e3
    rate_post = len(nrns[0].spikes['times']) / T * 1e3
    w_start   = cons[0][1].weights[0]
    w_end     = cons[0][1].weights[-1]
    w_change  = w_end - w_start
    direction = '⬆ potentiated' if w_change > 0.01 else ('⬇ depressed' if w_change < -0.01 else '↔ stable')

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Raster
    ax1 = axes[0]
    cs  = ['b', 'r']
    labels = ['Pre (blue)', 'Post (red)']
    for nrn, c, l, lbl in zip(nrns, cs, range(N), labels):
        rate = len(nrn.spikes['times']) / T * 1e3
        ax1.eventplot(nrn.spikes['times'], lineoffsets=2*l, colors=c,
                      label=f'{lbl}: {rate:.1f} Hz')
    ax1.set_title('Raster — Pre and Post neuron activity', fontsize=12, fontweight='bold')
    ax1.set_xlabel('Time (ms)')
    ax1.set_yticks([0, 2])
    ax1.set_yticklabels(['Pre\n(sender)', 'Post\n(receiver)'])
    ax1.set_xlim([0., T])
    ax1.legend(loc='upper right', fontsize=10)

    # Weight trace
    ax2 = axes[1]
    ax2.plot(np.arange(0., T, dt), cons[0][1].weights, c='purple',
             linewidth=2, label='Synaptic weight')
    for nrn, c in zip(nrns, cs):
        ax2.eventplot(nrn.spikes['times'], lineoffsets=w_start*0.85,
                      linelengths=w_start*0.1, colors=c, alpha=0.5)
    ax2.set_title(
        f'Synaptic weight over time\n'
        f'Start: {w_start:.2f}  →  End: {w_end:.2f}  |  {direction}',
        fontsize=11, fontweight='bold')
    ax2.set_xlabel('Time (ms)')
    ax2.set_ylabel('Synaptic weight (relative)')
    ax2.legend(fontsize=10)
    ax2.axhline(y=w_start, color='gray', linestyle=':', linewidth=1, alpha=0.7,
                label='Initial weight')

    plt.tight_layout()
    plt.show()

widgets.interact(
    update_rate_plasticity,
    J_ts=widgets.FloatSlider(min=0.01, max=2., step=0.01, value=1.0,
                              description='J_ts (start weight)',
                              style={'description_width':'initial'},
                              layout=widgets.Layout(width='400px')),
    rt=widgets.FloatSlider(min=30., max=70., step=1., value=50.,
                            description='Pre rate (rt)',
                            style={'description_width':'initial'},
                            layout=widgets.Layout(width='400px')),
    rs=widgets.FloatSlider(min=30., max=70., step=1., value=50.,
                            description='Post rate (rs)',
                            style={'description_width':'initial'},
                            layout=widgets.Layout(width='400px')),
    Itype=Itypes);

interactive(children=(FloatSlider(value=1.0, description='J_ts (start weight)', layout=Layout(width='400px'), …

### Explain — rate-dependent plasticity

**First: run with both rates equal (rt = rs = 50). Observe the weight trace.**

*Does the weight increase, decrease, or stay roughly stable? Does the result match your prediction?*

**My answer:**
```
[Write here]
```

---

**Now: increase Pre rate to 70, keep Post at 50.**

*What happens to the weight? Does the direction of change match Hebb's rule?*

**My answer:**
```
[Write here]
```

---

**Now: keep Pre at 70, reduce Post to 30.**

*How does the weight change compared to before? What does this tell you about which neuron's rate matters more?*

**My answer:**
```
[Write here]
```

---

**Now: systematically test all four input type combinations for one rate pair. Fill the table:**

| Pre input | Post input | Weight change direction | Weight change magnitude |
|-----------|-----------|------------------------|------------------------|
| Icur (regular) | Icur (regular) | | |
| Poisson (irregular) | Poisson (irregular) | | |
| Icur (regular) | Poisson (irregular) | | |
| Poisson (irregular) | Icur (regular) | | |

*Does the regularity of firing affect how much the weight changes? Why might this be?*

**My answer:**
```
[Write here]
```

---

### Real brain connection

Rate-dependent Hebbian plasticity is related to **Long-Term Potentiation (LTP)** and **Long-Term Depression (LTD)**, first demonstrated experimentally in the 1970s. LTP is triggered when both pre- and postsynaptic neurons are highly active together, and is widely believed to be a cellular mechanism underlying learning and memory.

*Think back to the degeneracy lab: you found that the STG can produce the pyloric rhythm with many different weight combinations. Rate-dependent plasticity could be one mechanism by which a circuit "finds" a working configuration over time — drifting weights until activity is appropriate. Does this idea make sense based on what you observed? What would the circuit need to do to stabilise its weights once a good configuration is found?*

**My answer:**
```
[Write here]
```

---

### Checkpoint 1

- Rate-dependent Hebbian plasticity says the synapse strengthens when: ______
- The synapse weakens when: ______
- Irregular (Poisson) firing affected weight change because: ______

## Part 2: Does timing matter?

### A flaw in Hebb's rule

The rate-dependent rule you just explored only cares about **how much** each neuron fires — not **when** exactly.

But think about what Hebb's original statement actually implies: neuron A fires, then neuron B fires as a *result* of A. The timing matters — A must fire *before* B for A to be considered a cause of B's firing.

This led to a more refined version of the rule discovered experimentally in the 1990s:

> **If the presynaptic neuron fires just before the postsynaptic neuron** — the synapse gets stronger (potentiation). The presynaptic neuron may have contributed to the postsynaptic neuron's spike.
>
> **If the presynaptic neuron fires just after the postsynaptic neuron** — the synapse gets weaker (depression). The presynaptic neuron arrived too late to have caused the spike.

The window of time over which this matters is on the order of **10–50 milliseconds**. Outside this window, the rule has no effect.

This is called **Spike-Timing-Dependent Plasticity (STDP)**.

---

### Why this is hard to study in a real animal

To map out the STDP rule experimentally, you need to:
- Record from two connected neurons simultaneously
- Control the precise relative timing of their spikes to millisecond accuracy
- Repeat this hundreds of times to measure the resulting weight change
- Do this across a range of timing differences from −50ms to +50ms

In real tissue, this is extraordinarily difficult — it requires dual patch-clamp recordings with precise current injection protocols, and each data point takes hours. A full STDP curve might take months in a real lab.

In simulation, you can explore the same curve in minutes. **This is computational modelling doing what biology cannot.**

---

### Reading the simulation

In this simulation, both neurons receive the same rate of input — what changes is **when** each neuron tends to fire relative to the other.

- **delay_pre** — delay on input to the presynaptic neuron (Pre, blue)
- **delay_post** — delay on input to the postsynaptic neuron (Post, red)

If delay_pre is small and delay_post is large, Pre fires first → this should potentiate.
If delay_pre is large and delay_post is small, Post fires first → this should depress.

The **purple weight trace** again shows whether the synapse is strengthening or weakening over the simulation.

---

### Predict — before running

*When delay_pre is small (Pre fires first), do you predict the synapse will be potentiated or depressed? Why?*

**My prediction:**
```
[Write here]
```

*What do you predict happens when both delays are equal (Pre and Post fire at roughly the same time)?*

**My prediction:**
```
[Write here]
```

*What do you predict happens when the timing difference is very large — say delay_pre = 1 and delay_post = 10?*

**My prediction:**
```
[Write here]
```

In [3]:
#@title ▶ Run to start spike-timing-dependent plasticity simulation
#@markdown The code is hidden by default

def update_spike_plasticity(delay_pre=1, delay_post=5, Itype='Icur'):
    h    = Simulator(dt=dt)
    nrns = [LIF(sim=h) for _ in range(N)]
    for nrn in nrns:
        nrn.update(tonic_neuron)

    rt = rs = 40.
    if Itype == 'Icur':
        noises = [Current_injector(sim=h, rate=r,
                                   start=int(T/dt*0.25), end=int(T/dt*0.75))
                  for r in [rt, rs]]
    elif Itype == 'Gaussian':
        noises = [Gaussian_generator(sim=h, mean=r, std=r,
                                     start=int(T/dt*0.25), end=int(T/dt*0.75))
                  for r in [rt, rs]]
    elif Itype == 'Poisson':
        noises = [Poisson_generator(sim=h, rate=r*3,
                                    start=int(T/dt*0.25), end=int(T/dt*0.75))
                  for r in [rt, rs]]

    for noise, nrn, delay in zip(noises, nrns, [delay_pre, delay_post]):
        nrn.connect(noise, {'ctype':'Static', 'weight':1e0, 'delay':delay})

    tps       = [['Static']*N for _ in range(N)]
    tps[0][1] = 'STDP'
    con       = np.array([[0., 1.0], [0., 0.]])
    dly       = np.random.uniform(2., 5., (N, N))
    synspecs  = [[{'ctype':tps[i][j], 'weight':con[i,j], 'delay':dly[i,j]}
                  for j in range(N)] for i in range(N)]
    cons = h.connect(nrns, nrns, synspecs)
    h.run(T)

    w_trace = cons[0][1].weights
    w_start = w_trace[0]
    w_end   = w_trace[-1]
    w_change = w_end - w_start
    timing_diff = delay_post - delay_pre

    if w_change > 0.01:
        direction = '⬆ Potentiated (Pre fires first — synapse strengthens)'
    elif w_change < -0.01:
        direction = '⬇ Depressed (Post fires first — synapse weakens)'
    else:
        direction = '↔ Stable (no clear change)'

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Raster
    ax1 = axes[0]
    cs  = ['b', 'r']
    for nrn, c, l, lbl in zip(nrns, cs, range(N),
                               ['Pre (blue — sender)', 'Post (red — receiver)']):
        rate = len(nrn.spikes['times']) / T * 1e3
        ax1.eventplot(nrn.spikes['times'], lineoffsets=2*l, colors=c,
                      label=f'{lbl}: {rate:.1f} Hz')
    ax1.set_title(
        f'Raster — delay_pre={delay_pre}ms, delay_post={delay_post}ms\n'
        f'Effective timing advantage: Pre fires ~{timing_diff}ms earlier',
        fontsize=11, fontweight='bold')
    ax1.set_xlabel('Time (ms)')
    ax1.set_yticks([0, 2])
    ax1.set_yticklabels(['Pre\n(sender)', 'Post\n(receiver)'])
    ax1.set_xlim([0., T])
    ax1.legend(loc='upper right', fontsize=10)

    # Weight trace
    ax2 = axes[1]
    ax2.plot(np.arange(0., T, dt), w_trace, c='purple',
             linewidth=2, label='Synaptic weight')
    for nrn, c in zip(nrns, cs):
        ax2.eventplot(nrn.spikes['times'], lineoffsets=w_start*0.85,
                      linelengths=w_start*0.1, colors=c, alpha=0.5)
    ax2.axhline(y=w_start, color='gray', linestyle=':', linewidth=1,
                alpha=0.7, label='Initial weight')
    ax2.set_title(
        f'Synaptic weight over time\n'
        f'Start: {w_start:.2f}  →  End: {w_end:.2f}\n{direction}',
        fontsize=11, fontweight='bold')
    ax2.set_xlabel('Time (ms)')
    ax2.set_ylabel('Synaptic weight (relative)')
    ax2.legend(fontsize=10)

    plt.tight_layout()
    plt.show()

widgets.interact(
    update_spike_plasticity,
    delay_pre=widgets.IntSlider(min=1, max=10, step=1, value=1,
                                 description='delay_pre (ms)',
                                 style={'description_width':'initial'},
                                 layout=widgets.Layout(width='400px')),
    delay_post=widgets.IntSlider(min=1, max=10, step=1, value=5,
                                  description='delay_post (ms)',
                                  style={'description_width':'initial'},
                                  layout=widgets.Layout(width='400px')),
    Itype=Itypes);

interactive(children=(IntSlider(value=1, description='delay_pre (ms)', layout=Layout(width='400px'), max=10, m…

### Explain — spike-timing-dependent plasticity

**First run: delay_pre = 1, delay_post = 5 (Pre fires first)**

*What happened to the weight? Does this match your prediction?*

**My answer:**
```
[Write here]
```

---

**Now reverse it: delay_pre = 5, delay_post = 1 (Post fires first)**

*What happens now? How does the direction of change compare to the previous run?*

**My answer:**
```
[Write here]
```

---

**Now systematically vary the timing. Fill in the table below:**

Keep delay_post = 5 constant and vary delay_pre:

| delay_pre | delay_post | Pre fires first? | Weight change |
|-----------|-----------|-----------------|---------------|
| 1 | 5 | Yes | |
| 3 | 5 | Yes | |
| 5 | 5 | Same time | |
| 7 | 5 | No | |
| 10 | 5 | No | |

*Does the magnitude of weight change depend on how far apart the spikes are in time? What happens as the timing difference approaches zero?*

**My answer:**
```
[Write here]
```

---

**Now switch to Poisson input (irregular firing)**

*With the same delay settings, does the STDP rule behave differently when firing is irregular? Why might noisy timing affect the plasticity outcome?*

**My answer:**
```
[Write here]
```

---

### Mapping the STDP curve

You have just mapped out — by hand — the general shape of what neuroscientists call the **STDP learning window**. It looks roughly like this:

- For small positive timing differences (Pre just before Post): strong potentiation
- For small negative timing differences (Post just before Pre): strong depression
- For large timing differences in either direction: little or no change

This asymmetric curve was measured experimentally in 1997–1998 by Bi & Poo and others, and it required years of painstaking dual patch-clamp recordings.

*Sketch or describe the shape of the curve you observed in your own words — how does weight change vary with timing difference?*

**My answer:**
```
[Write here]
```

---

### Real brain connection

STDP has been found in the cortex, hippocampus, cerebellum, and many other brain regions. It is thought to be a mechanism by which neural circuits learn the **causal relationships** between events — if A consistently happens before B, the connection from A to B strengthens because A may be causing B.

*Think about Hebb's original 1949 statement again: "neuron A fires and repeatedly takes part in firing neuron B." STDP is a more precise version of this — it only strengthens connections where Pre fires before Post (and could therefore have contributed to the postsynaptic spike). In what sense is STDP more "intelligent" than simple rate-dependent plasticity?*

**My answer:**
```
[Write here]
```

*In the pyloric circuit, neurons fire in a strict sequence: AB/PD → LP → PY. If STDP were operating in this circuit, which synapses would you predict would be potentiated and which would be depressed?*

**My answer:**
```
[Write here]
```

---

### Checkpoint 2

- In STDP, the synapse is potentiated when: ______
- The synapse is depressed when: ______
- The key difference between STDP and rate-dependent Hebbian plasticity is: ______
- Irregular (Poisson) firing affected STDP because: ______

## Part 3: Putting it together — plasticity and circuit stability

You have now seen two forms of synaptic plasticity. Both implement versions of Hebb's rule:
- **Rate-dependent**: neurons that are both active together → synapse strengthens
- **STDP**: pre-synaptic neuron fires just before post-synaptic → synapse strengthens

Both rules have a problem: if synapses always strengthen when neurons are co-active, what stops all synapses from growing without bound? A brain where everything gets stronger and stronger would eventually fall into a state of constant maximal activity — epilepsy.

Real plasticity rules include a **depression** mechanism alongside potentiation, and the balance between the two depends on activity levels. This is part of a broader process called **homeostatic plasticity** — the brain's way of keeping activity in a useful range by adjusting its own weights.

---

### Connecting back to degeneracy

In the degeneracy lab you found that the pyloric circuit can produce its rhythm with many different weight combinations. Plasticity is one explanation for *how* a real animal might navigate that landscape:

- Weights start at some value (determined by genetics and development)
- Activity-dependent plasticity gradually adjusts them
- Homeostatic mechanisms keep the overall activity level stable
- The circuit settles into one of the many valid configurations you found in the random search

This is why no two lobsters need to have identical synaptic weights — each one finds its own path through the degenerate landscape.

---

### Final reflection

*In your own words, explain the difference between rate-dependent Hebbian plasticity and STDP. When might one be more appropriate than the other?*

**My answer:**
```
[Write here]
```

*If plasticity is always running in a circuit, how might a circuit ever be stable? What would need to be true for a set of weights to stop changing?*

**My answer:**
```
[Write here]
```

*You mapped the STDP curve in minutes using simulation. Bi & Poo took years to map it experimentally. What does this tell you about the role of computational modelling in neuroscience — and what are its limits?*

**My answer:**
```
[Write here]
```

---

### Final Checkpoint

- Hebb's rule in one sentence: ______
- STDP is more precise than rate-dependent plasticity because: ______
- The problem with unconstrained potentiation is: ______
- Plasticity could explain degeneracy in the STG because: ______

---

### Bonus — Think deeper

*STDP is asymmetric: pre-before-post causes potentiation but post-before-pre causes depression. This means the rule has a built-in sense of "direction" — it strengthens forward connections (A causes B) and weakens backward ones (B precedes A coincidentally). What does this suggest about what STDP is optimising for in a neural circuit?*

**My answer:**
```
[Write here]
```

*The pyloric rhythm fires in a precise sequence: AB/PD → LP → PY. If STDP operates in this circuit, it would selectively strengthen connections that go in the direction of the rhythm and weaken those that go against it. Could STDP be part of how the circuit learns or maintains its rhythm? What experiment would test this?*

**My answer:**
```
[Write here]
```